In [8]:
import sys
import os
import psycopg2
from dotenv import load_dotenv

sys.path.append(os.path.abspath(".."))

load_dotenv()
url = os.getenv("DATABASE_URL")

In [9]:
conn = psycopg2.connect(url, sslmode="require")
cur = conn.cursor()

In [10]:
query = """
SELECT 
    ticker,
    CASE 
        WHEN EXTRACT(DOW FROM MAX(timestamp)) = 3 THEN MAX(timestamp)
        ELSE NULL
    END AS expiry_date
FROM orderbooks
GROUP BY ticker
"""
cur.execute(query)
rows = cur.fetchall()

data = {ticker: expiry for ticker, expiry in rows}
tickers = [ticker for ticker, date in data.items()]

In [11]:
from datetime import datetime, timezone

#sorted by expiry dates
data = dict(sorted(data.items(), key=lambda item: item[1] if item[1] is not None else datetime(2099,1,1, tzinfo=timezone.utc)))

In [12]:
expiry_dates = [date.replace(hour=0, minute=0, second=0, microsecond=0) for ticker, date in data.items() if date is not None]
unique_dates = list(dict.fromkeys(expiry_dates))
print(unique_dates)

[datetime.datetime(2026, 3, 25, 0, 0, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 4, 1, 0, 0, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 4, 8, 0, 0, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 4, 15, 0, 0, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 4, 22, 0, 0, tzinfo=datetime.timezone.utc)]


In [42]:
import pandas as pd
for date in unique_dates:
    
    #not expiring this day 
    not_expiring_tickers = [ticker for ticker, expiry_date in data.items() 
                             if (expiry_date != None and expiry_date.replace(hour=0, minute=0, second=0, microsecond=0) > date)  
                             or expiry_date == None]

    query = """
        SELECT * FROM (
            (
                SELECT DISTINCT ON (ticker) *
                FROM orderbooks
                WHERE timestamp >= %s AND timestamp < %s + INTERVAL '1 day'
                AND ticker = ANY(%s)
                ORDER BY ticker, timestamp ASC
            )
            UNION ALL
            (
                SELECT DISTINCT ON (ticker) *
                FROM orderbooks
                WHERE timestamp >= %s AND timestamp < %s + INTERVAL '1 day'
                AND ticker = ANY(%s)
                ORDER BY ticker, timestamp DESC
            )
        ) AS combined
        ORDER BY ticker, timestamp;  -- или любая другая сортировка
    """

    cur.execute(query, (date, date, not_expiring_tickers, date, date, not_expiring_tickers))
    rows = cur.fetchall()

    col_names = [desc[0] for desc in cur.description]
    df = pd.DataFrame(rows, columns=col_names)


    
    df_first = df.groupby('ticker').first().reset_index()  
    df_last = df.groupby('ticker').last().reset_index()
    df = df_first.merge(df_last, on='ticker', suffixes=('_first', '_last'))

    df['best_bid_first'] = df['bids_first'].apply(lambda x: max(x, key=lambda y: y['price'])['price'] if x else None)
    df['best_ask_first'] = df['asks_first'].apply(lambda x: min(x, key=lambda y: y['price'])['price'] if x else None)
    df['best_bid_last'] = df['bids_last'].apply(lambda x: max(x, key=lambda y: y['price'])['price'] if x else None)
    df['best_ask_last'] = df['asks_last'].apply(lambda x: min(x, key=lambda y: y['price'])['price'] if x else None)

    
    break
    
    

      id     ticker class_code                 timestamp  \
0  51203   SR270CD6    OPTSPOT 2026-03-25 06:00:45+00:00   
1  63763   SR270CD6    OPTSPOT 2026-03-25 21:07:23+00:00   
2  53231  SR270CD6A    OPTSPOT 2026-03-25 07:01:28+00:00   
3  57097  SR270CD6A    OPTSPOT 2026-03-25 12:02:03+00:00   
4  51192   SR270CP6    OPTSPOT 2026-03-25 06:00:15+00:00   

                                  bids  \
0  [{'price': 50.33, 'quantity': 800}]   
1    [{'price': 5.15, 'quantity': 10}]   
2  [{'price': 48.22, 'quantity': 800}]   
3                                   []   
4                                   []   

                                                asks  bid_volume  ask_volume  
0  [{'price': 50.5, 'quantity': 664}, {'price': 5...         800        1464  
1                                                 []          10           0  
2                [{'price': 49.61, 'quantity': 800}]         800         800  
3                 [{'price': 57.84, 'quantity': 83}]           0      